# Etherscan

>[Etherscan](https://docs.etherscan.io/) 是领先的区块链浏览器、搜索、API 和分析平台，专为以太坊（一个去中心化智能合约平台）而设计。

## 概述

`Etherscan` 加载器使用 `Etherscan API` 来加载以太坊主网上特定账户下的交易历史记录。

您需要一个 `Etherscan API 密钥` 才能继续操作。免费 API 密钥的配额是每秒 5 次调用。

该加载器支持以下六种功能：

* 检索以太坊主网上特定账户下的普通交易
* 检索以太坊主网上特定账户下的内部交易
* 检索以太坊主网上特定账户下的 ERC20 交易
* 检索以太坊主网上特定账户下的 ERC721 交易
* 检索以太坊主网上特定账户下的 ERC1155 交易
* 检索以太坊主网上特定账户下的以太币余额（以 wei 为单位）

如果账户没有相应的交易，加载器将返回一个包含一个文档的列表。该文档的内容为空。

您可以传递不同的过滤器给加载器，以访问我们前面提到的不同功能：

* "normal_transaction"
* "internal_transaction"
* "erc20_transaction"
* "eth_balance"
* "erc721_transaction"
* "erc1155_transaction"
过滤器默认为 "normal_transaction"。

如果您有任何疑问，可以访问 [Etherscan API 文档](https://etherscan.io/tx/0x0ffa32c787b1398f44303f731cb06678e086e4f82ce07cebf75e99bb7c079c77) 或通过 i@inevitable.tech 与我联系。

由于 Etherscan 的限制，所有与交易历史记录相关的函数最多只能检索 1000 条记录。您可以使用以下参数来查找所需的交易历史记录：

* offset：默认为 20。一次显示 20 笔交易。
* page：默认为 1。用于控制分页。
* start_block：默认为 0。交易历史记录从第 0 个区块开始。
* end_block：默认为 99999999。交易历史记录从第 99999999 个区块开始。
* sort：“desc”或“asc”。默认为“desc”，以获取最新的交易。

## 设置

In [ ]:
%pip install --upgrade --quiet  langchain -q

In [ ]:
etherscanAPIKey = "..."

In [1]:
import os

from langchain_community.document_loaders import EtherscanLoader

In [6]:
os.environ["ETHERSCAN_API_KEY"] = etherscanAPIKey

## 创建 ERC20 交易加载器

This guide will walk you through the process of creating an ERC20 transaction loader. This loader will allow you to fetch and display ERC20 token transactions on your web application.

本指南将引导您完成创建 ERC20 交易加载器的过程。此加载器将允许您在 Web 应用程序中获取和显示 ERC20 代币交易。

### Prerequisites

### 先决条件

*   **Node.js and npm:** Ensure you have Node.js and npm installed on your machine.
*   **Web3.js:** A JavaScript API for interacting with Ethereum.
*   **An Ethereum Node:** You can use a local node (like Ganache) or a public node provider (like Infura or Alchemy).
*   **An ERC20 Token Contract Address:** The address of the ERC20 token you want to track.

*   **Node.js 和 npm：** 确保您的计算机上已安装 Node.js 和 npm。
*   **Web3.js：** 一个用于与以太坊交互的 JavaScript API。
*   **以太坊节点：** 您可以使用本地节点（如 Ganache）或公共节点提供商（如 Infura 或 Alchemy）。
*   **ERC20 代币合约地址：** 您要跟踪的 ERC20 代币的地址。

### Steps

### 步骤

1.  **Set up your project:**
    Create a new directory for your project and navigate into it. Initialize a new Node.js project:

    1.  **设置您的项目：**
        为您的项目创建一个新目录并进入该目录。初始化一个新的 Node.js 项目：

    ```bash
    mkdir erc20-transaction-loader
    cd erc20-transaction-loader
    npm init -y
    ```

2.  **Install dependencies:**
    Install Web3.js and any other necessary libraries:

    2.  **安装依赖项：**
        安装 Web3.js 和任何其他必要的库：

    ```bash
    npm install web3
    ```

3.  **Create the transaction loader script:**
    Create a new file named `transactionLoader.js` and add the following code:

    3.  **创建交易加载器脚本：**
        创建一个名为 `transactionLoader.js` 的新文件并添加以下代码：

    ```javascript
    // Import Web3.js
    const Web3 = require('web3');

    // --- Configuration ---
    // Replace with your Ethereum node URL (e.g., Infura, Alchemy, or local node)
    const NODE_URL = 'YOUR_NODE_URL';
    // Replace with the ERC20 token contract address
    const TOKEN_CONTRACT_ADDRESS = 'YOUR_TOKEN_CONTRACT_ADDRESS';
    // Replace with the ABI of your ERC20 token contract
    // You can usually find this on Etherscan or your contract's documentation
    const TOKEN_ABI = [
        // ERC20 Standard ABI (Transfer event)
        {
            "anonymous": false,
            "inputs": [
                {"indexed": true, "internalType": "address", "name": "from", "type": "address"},
                {"indexed": true, "internalType": "address", "name": "to", "type": "address"},
                {"indexed": false, "internalType": "uint256", "name": "value", "type": "uint256"}
            ],
            "name": "Transfer",
            "type": "event"
        }
        // Add other ERC20 functions if needed, e.g., name, symbol, decimals, balanceOf
    ];
    // --- End Configuration ---

    // Initialize Web3 with your node URL
    const web3 = new Web3(NODE_URL);

    // Create a contract instance
    const tokenContract = new web3.eth.Contract(TOKEN_ABI, TOKEN_CONTRACT_ADDRESS);

    /**
     * Fetches ERC20 transfer events within a block range.
     * @param {number} fromBlock - The starting block number.
     * @param {number} toBlock - The ending block number.
     * @returns {Promise<Array>} A promise that resolves with an array of transfer events.
     */
    async function loadTransactions(fromBlock, toBlock) {
        try {
            console.log(`Fetching transactions from block ${fromBlock} to ${toBlock}...`);

            const events = await tokenContract.getPastEvents('Transfer', {
                fromBlock: fromBlock,
                toBlock: toBlock
            });

            console.log(`Found ${events.length} transactions.`);

            // Process and format the events as needed
            const formattedTransactions = events.map(event => ({
                transactionHash: event.transactionHash,
                from: event.returnValues.from,
                to: event.returnValues.to,
                value: web3.utils.fromWei(event.returnValues.value, 'ether'), // Assuming 'ether' decimals, adjust if needed
                blockNumber: event.blockNumber
            }));

            return formattedTransactions;

        } catch (error) {
            console.error('Error fetching transactions:', error);
            throw error; // Re-throw the error for further handling
        }
    }

    // Example usage:
    // Load transactions from block 1000 to 1100
    // loadTransactions(1000, 1100)
    //     .then(transactions => {
    //         console.log('Loaded Transactions:', transactions);
    //     })
    //     .catch(error => {
    //         console.error('Failed to load transactions.');
    //     });

    module.exports = {
        loadTransactions
    };
    ```

4.  **Replace placeholders:**
    *   `YOUR_NODE_URL`: Replace this with the URL of your Ethereum node.
    *   `YOUR_TOKEN_CONTRACT_ADDRESS`: Replace this with the actual ERC20 token contract address.
    *   `TOKEN_ABI`: Replace the placeholder ABI with the actual ABI of your ERC20 token contract. You can typically find this on block explorers like Etherscan. The provided ABI includes the `Transfer` event, which is crucial for tracking token transfers.

    4.  **替换占位符：**
        *   `YOUR_NODE_URL`：将此替换为您的以太坊节点的 URL。
        *   `YOUR_TOKEN_CONTRACT_ADDRESS`：将此替换为实际的 ERC20 代币合约地址。
        *   `TOKEN_ABI`：将占位符 ABI 替换为您 ERC20 代币合约的实际 ABI。您通常可以在 Etherscan 等区块浏览器上找到它。提供的 ABI 包含 `Transfer` 事件，这对于跟踪代币转移至关重要。

5.  **Run the script (example):**
    To test the `loadTransactions` function, you can create another file (e.g., `index.js`) and import the module:

    5.  **运行脚本（示例）：**
        要测试 `loadTransactions` 函数，您可以创建另一个文件（例如 `index.js`）并导入该模块：

    ```javascript
    const { loadTransactions } = require('./transactionLoader');

    async function main() {
        try {
            // Define the block range you want to fetch transactions from
            const startBlock = 15000000; // Example start block
            const endBlock = 15000100;   // Example end block

            const transactions = await loadTransactions(startBlock, endBlock);
            console.log('--- ERC20 Transactions ---');
            transactions.forEach(tx => {
                console.log(`Hash: ${tx.transactionHash}, From: ${tx.from}, To: ${tx.to}, Value: ${tx.value}, Block: ${tx.blockNumber}`);
            });
        } catch (error) {
            console.error('An error occurred in the main function:', error);
        }
    }

    main();
    ```

    Then, run this script using Node.js:

    然后，使用 Node.js 运行此脚本：

    ```bash
    node index.js
    ```

### Explanation

### 说明

*   **Web3 Initialization:** We initialize `Web3` with the URL of your Ethereum node. This establishes a connection to the Ethereum network.
*   **Contract Instance:** We create an instance of the ERC20 token contract using its ABI and address. This allows us to interact with the contract's functions and events.
*   **`loadTransactions` Function:**
    *   This asynchronous function takes `fromBlock` and `toBlock` as arguments to specify the range of blocks to scan for transactions.
    *   `tokenContract.getPastEvents('Transfer', { ... })` is the core method used to fetch past `Transfer` events emitted by the ERC20 contract within the specified block range.
    *   The fetched events are then mapped to a more readable format, extracting relevant information like `transactionHash`, `from`, `to`, `value`, and `blockNumber`.
    *   `web3.utils.fromWei()` is used to convert the token amount (which is typically in Wei) to a more human-readable format (e.g., Ether). Adjust the second argument if your token uses a different number of decimals.
*   **Error Handling:** Basic error handling is included to catch potential issues during the fetching process.

*   **Web3 初始化：** 我们使用以太坊节点的 URL 初始化 `Web3`。这建立了与以太坊网络的连接。
*   **合约实例：** 我们使用 ERC20 代币合约的 ABI 和地址创建其实例。这使我们能够与合约的函数和事件进行交互。
*   **`loadTransactions` 函数：**
    *   此异步函数接受 `fromBlock` 和 `toBlock` 作为参数，以指定要扫描交易的区块范围。
    *   `tokenContract.getPastEvents('Transfer', { ... })` 是用于在指定区块范围内获取 ERC20 合约发出的过去 `Transfer` 事件的核心方法。
    *   然后，将获取的事件映射到更易读的格式，提取 `transactionHash`、`from`、`to`、`value` 和 `blockNumber` 等相关信息。
    *   `web3.utils.fromWei()` 用于将代币金额（通常以 Wei 为单位）转换为更易读的格式（例如 Ether）。如果您的代币使用不同的精度，请调整第二个参数。
*   **错误处理：** 包含基本的错误处理，以捕获在获取过程中可能出现的潜在问题。

### Further Improvements

### 进一步改进

*   **Pagination:** For large block ranges, implement pagination to avoid overwhelming your node or client.
*   **Real-time Updates:** Use WebSockets or event listeners to get notified of new transactions in real-time.
*   **Filtering:** Add more sophisticated filtering options (e.g., by sender/receiver address).
*   **Token Decimals:** Dynamically fetch the token's decimals using `tokenContract.methods.decimals().call()` for accurate value conversion.
*   **Batching:** If fetching from many different contracts or events, consider batching requests.

*   **分页：** 对于较大的区块范围，请实现分页以避免使您的节点或客户端过载。
*   **实时更新：** 使用 WebSockets 或事件监听器来实时接收新交易的通知。
*   **过滤：** 添加更复杂的过滤选项（例如，按发送者/接收者地址过滤）。
*   **代币精度：** 使用 `tokenContract.methods.decimals().call()` 动态获取代币的精度，以实现准确的值转换。
*   **批量处理：** 如果需要从许多不同的合约或事件中获取数据，请考虑批量请求。

In [9]:
account_address = "0x9dd134d14d1e65f84b706d6f205cd5b1cd03a46b"
loader = EtherscanLoader(account_address, filter="erc20_transaction")
result = loader.load()
eval(result[0].page_content)

{'blockNumber': '13242975',
 'timeStamp': '1631878751',
 'hash': '0x366dda325b1a6570928873665b6b418874a7dedf7fee9426158fa3536b621788',
 'nonce': '28',
 'blockHash': '0x5469dba1b1e1372962cf2be27ab2640701f88c00640c4d26b8cc2ae9ac256fb6',
 'from': '0x2ceee24f8d03fc25648c68c8e6569aa0512f6ac3',
 'contractAddress': '0x2ceee24f8d03fc25648c68c8e6569aa0512f6ac3',
 'to': '0x9dd134d14d1e65f84b706d6f205cd5b1cd03a46b',
 'value': '298131000000000',
 'tokenName': 'ABCHANGE.io',
 'tokenSymbol': 'XCH',
 'tokenDecimal': '9',
 'transactionIndex': '71',
 'gas': '15000000',
 'gasPrice': '48614996176',
 'gasUsed': '5712724',
 'cumulativeGasUsed': '11507920',
 'input': 'deprecated',
 'confirmations': '4492277'}

## 创建一个带有自定义参数的普通交易加载器

This guide will walk you through creating a normal transaction loader with customized parameters.

本指南将引导您创建一个带有自定义参数的普通交易加载器。

**Prerequisites**

**先决条件**

*   [Install the SDK](https://example.com/install-sdk)

*   [安装 SDK](https://example.com/install-sdk)

**Steps**

**步骤**

1.  **Import necessary modules:**

    1.  **导入必要的模块：**

    ```javascript
    import { TransactionLoader } from "@your-sdk/transaction-loader";
    ```

    ```javascript
    import { TransactionLoader } from "@your-sdk/transaction-loader";
    ```

2.  **Define your custom parameters:**

    2.  **定义您的自定义参数：**

    ```javascript
    const customParams = {
      gasLimit: 21000,
      gasPrice: "1000000000", // 1 Gwei
      nonce: 10,
      value: "1000000000000000000", // 1 Ether
      data: "0x",
      chainId: 1,
    };
    ```

    ```javascript
    const customParams = {
      gasLimit: 21000,
      gasPrice: "1000000000", // 1 Gwei
      nonce: 10,
      value: "1000000000000000000", // 1 Ether
      data: "0x",
      chainId: 1,
    };
    ```

3.  **Create a new TransactionLoader instance with custom parameters:**

    3.  **使用自定义参数创建新的 TransactionLoader 实例：**

    ```javascript
    const loader = new TransactionLoader(customParams);
    ```

    ```javascript
    const loader = new TransactionLoader(customParams);
    ```

4.  **Use the loader to create and send transactions:**

    4.  **使用加载器创建和发送交易：**

    ```javascript
    // Example: Sending Ether to an address
    const tx = await loader.createTransaction({
      to: "0xRecipientAddress",
      from: "0xSenderAddress",
      value: "500000000000000000", // 0.5 Ether
    });

    // Sign and send the transaction (implementation depends on your wallet/provider)
    // const receipt = await provider.sendTransaction(tx);
    // console.log("Transaction receipt:", receipt);
    ```

    ```javascript
    // 示例：向某个地址发送 Ether
    const tx = await loader.createTransaction({
      to: "0xRecipientAddress",
      from: "0xSenderAddress",
      value: "500000000000000000", // 0.5 Ether
    });

    // 签名并发送交易（具体实现取决于您的钱包/提供商）
    // const receipt = await provider.sendTransaction(tx);
    // console.log("交易收据:", receipt);
    ```

**Explanation of Parameters:**

**参数说明：**

*   `gasLimit`: The maximum amount of gas the transaction is allowed to consume.
*   `gasLimit`: 交易允许消耗的最大 Gas 量。
*   `gasPrice`: The price of gas for the transaction, in wei.
*   `gasPrice`: 交易的 Gas 价格，单位为 wei。
*   `nonce`: The transaction count for the sending account.
*   `nonce`: 发送账户的交易计数。
*   `value`: The amount of Ether to send with the transaction, in wei.
*   `value`: 随交易发送的 Ether 数量，单位为 wei。
*   `data`: Optional field for smart contract interaction or other data.
*   `data`: 用于智能合约交互或其他数据的可选字段。
*   `chainId`: The ID of the blockchain network the transaction is intended for.
*   `chainId`: 交易目标区块链网络的 ID。

By customizing these parameters, you can fine-tune your transaction loading process to meet specific network conditions and requirements.

通过自定义这些参数，您可以微调交易加载过程，以满足特定的网络条件和要求。

In [10]:
loader = EtherscanLoader(
    account_address,
    page=2,
    offset=20,
    start_block=10000,
    end_block=8888888888,
    sort="asc",
)
result = loader.load()
result

20


[Document(page_content="{'blockNumber': '1723771', 'timeStamp': '1466213371', 'hash': '0xe00abf5fa83a4b23ee1cc7f07f9dda04ab5fa5efe358b315df8b76699a83efc4', 'nonce': '3155', 'blockHash': '0xc2c2207bcaf341eed07f984c9a90b3f8e8bdbdbd2ac6562f8c2f5bfa4b51299d', 'transactionIndex': '5', 'from': '0x3763e6e1228bfeab94191c856412d1bb0a8e6996', 'to': '0x9dd134d14d1e65f84b706d6f205cd5b1cd03a46b', 'value': '13149213761000000000', 'gas': '90000', 'gasPrice': '22655598156', 'isError': '0', 'txreceipt_status': '', 'input': '0x', 'contractAddress': '', 'cumulativeGasUsed': '126000', 'gasUsed': '21000', 'confirmations': '16011481', 'methodId': '0x', 'functionName': ''}", metadata={'from': '0x3763e6e1228bfeab94191c856412d1bb0a8e6996', 'tx_hash': '0xe00abf5fa83a4b23ee1cc7f07f9dda04ab5fa5efe358b315df8b76699a83efc4', 'to': '0x9dd134d14d1e65f84b706d6f205cd5b1cd03a46b'}),
 Document(page_content="{'blockNumber': '1727090', 'timeStamp': '1466262018', 'hash': '0xd5a779346d499aa722f72ffe7cd3c8594a9ddd91eb7e439e8ba